In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Introduction

This notebook contains code for a micro-Small Language Model (< 2 Million parameters) consisting of a Language Model with 1-2 transformer blocks and multi-headed, causal self-attention. This model is pretrained on the TinyStories dataset for the next-token prediction task. The pretrained model is then repeatedly input generated text to generate a short story.

# Steps:
1. **Data Exploration**
2. **Data Preprocessing**
     1. Tokenization
     2. Saving data to memory
     3. Input-ouput pairing
     4. Projection to Token Embedding Space
     5. Projection to Positional Embedding Space (for later; unclear on this)
3. **Building Model Architecture**
    1. Attention
    2. Feed-forward Network/ Multi-layer Perceptrons
    3. Layer Normalization
    4. Transformer
    5. Small Language Model (contains embedding projection, transformer blocks, LM Head, etc.)
4. **Pre-training**
    1. Defining cross-entropy loss function
    2. Updating parameters using backpropagation
    3. Pre-training using training data
    4. Validate model on validation dataset
    5. Visualize and analyze training and validation losses
    6. Hyperparameter tuning and repeat training and validation
5. **Short Story Generation**
6. **Explore and Evaluate Generated Output**


# Step 1: Data Exploration

In [ ]:
if not os.path.exists('./data'):
    os.makedirs('./data')
    
kagglehub.dataset_download('thedevastator/tinystories-narrative-classification',
                           path = 'train.csv', output_dir = './data')
kagglehub.dataset_download('thedevastator/tinystories-narrative-classification',
                           path = 'validation.csv', output_dir = './data')

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/train.csv')
val = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv')

In [ ]:
pd.options.display.max_rows = 500

print("Summary of training dataset:")
print(train.describe)

In [ ]:
print("Summary of validation dataset:")
print(val.describe)

In [ ]:
print('First story in training set:')
print(train.iloc[0]['text'])

## Select sample of 10 stories for initial run 

In [ ]:
print('2 more stories from training set and validation set each:')
train_sample = train.sample(10, random_state = 1)
val_sample = val.sample(2, random_state = 1)

print(train_sample.iloc[0]['text'])
print('-------------X-------------')
print(train_sample.iloc[1]['text'])
print('-------------X-------------')
print(val_sample.iloc[0]['text'])
print('-------------X-------------')
print(val_sample.iloc[1]['text'])

## Tokenize data 

In [ ]:
!pip install git+https://github.com/huggingface/transformers

In [ ]:

from transformers import GPT2Tokenizer, BatchEncoding

tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2')
print('Vocabulary size = ', tokenizer.vocab_size)

trainTokens = tokenizer(train_sample['text'].tolist())
valTokens = tokenizer(val_sample['text'].tolist())

print(trainTokens[0])
print('---------------X--------------')
print(valTokens[0])
print('---------------X--------------')
print(trainTokens['input_ids'][0])

## Create input-output pairs 

For the next token prediction task, a **sliding window** is defined, such that given the tokens in the window, i.e. the **context**, the next token in the sequence is considered to be the expected output. The max number of tokens that can be in the window is the **context size**. Tokenized data is grouped into these input (the tokens in the window) and output (the next expected token) pairs. These groups are called **chunks**.

In [ ]:
def io_pairing(document, contextSize):
    inputTokens = np.zeros((len(document) - contextSize, contextSize))
    # Leave out the last token;it is never input to the model
    for i in range(len(document) - contextSize):
        inputTokens[i,:] = document[i: i + contextSize]
    outputTokens = np.array(document[contextSize: len(document)])
    return inputTokens, outputTokens

In [ ]:
inputTokens, outputTokens = io_pairing(document = trainTokens['input_ids'][0], contextSize = 4)
print(inputTokens[0:4,:])
print(outputTokens[0:4])
print(inputTokens.shape)
print(outputTokens.shape)

In [ ]:
def get_io_pairs(data,contextSize):
    X, Y = [], []
    for document in data['input_ids']:
        inputTokens, outputTokens = io_pairing(document = data['input_ids'][0], contextSize = 4)
        X.append(inputTokens)
        Y.append(outputTokens)
print(len(trainX))
print(trainY[0:4])